In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 105.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 90.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [2]:
!touch data.yml

In [3]:
%%writefile data.yml
nc: 1
names: ['Nodules']

train: /kaggle/input/lung-nodule-detection/ct_images/images/train
val: /kaggle/input/lung-nodule-detection/ct_images/images/val

Overwriting data.yml


In [4]:
import os
import yaml
from ultralytics import YOLO
import itertools
import pandas as pd
from datetime import datetime
import torch

class YOLOHyperparameterTuner:
    def __init__(self, data_path, device=0, workers=0):
        self.data_path = data_path
        self.device = device
        self.results_log = []
        self.workers = workers
    def get_optimized_hyperparameters(self):
        """
        VALIDATED hyperparameters from official Ultralytics YOLO documentation
        Only includes parameters that are officially supported
        """
        return {
            # Model Architecture Parameters
            'models': {
                'yolov8': ['yolov8n.pt', 'yolov8s.pt', 'yolov8m.pt'],
                'yolov10': ['yolov10n.pt', 'yolov10s.pt', 'yolov10m.pt'],
                'yolov11': ['yolo11n.pt', 'yolo11s.pt', 'yolo11m.pt']
            },
            
            # ✅ CORE TRAINING PARAMETERS (Officially Documented)
            'epochs': [100, 150, 200],        # int, default=100
            'batch': [8, 16, 32],             # int, default=16 (CORRECTED: was batch_size)
            'imgsz': [640, 832, 1024],        # int or list, default=640
            'lr0': [0.001, 0.01, 0.02],       # float, default=0.01 (initial learning rate)
            'lrf': [0.01, 0.1, 0.2],          # float, default=0.01 (final lr factor)
            'momentum': [0.9, 0.937, 0.95],   # float, default=0.937
            'weight_decay': [0.0005, 0.001],  # float, default=0.0005
            
            # ✅ OPTIMIZER OPTIONS (Officially Documented)
            'optimizer': ['SGD', 'Adam', 'AdamW', 'NAdam', 'RAdam', 'RMSProp', 'auto'],
            
            # ✅ AUGMENTATION PARAMETERS (Officially Documented with Ranges)
            'hsv_h': [0.015, 0.025],          # float, default=0.015, range=0.0-1.0
            'hsv_s': [0.5, 0.7],              # float, default=0.7, range=0.0-1.0  
            'hsv_v': [0.2, 0.4],              # float, default=0.4, range=0.0-1.0
            'degrees': [0.0, 5.0, 10.0],      # float, default=0.0, range=0.0-180
            'translate': [0.1, 0.2],          # float, default=0.1, range=0.0-1.0
            'scale': [0.3, 0.5],              # float, default=0.5, range=>=0.0
            'shear': [0.0, 2.0, 5.0],         # float, default=0.0, range=-180 to +180
            'perspective': [0.0, 0.0001, 0.0005],  # float, default=0.0, range=0.0-0.001
            'flipud': [0.0, 0.2],             # float, default=0.0, range=0.0-1.0
            'fliplr': [0.3, 0.5],             # float, default=0.5, range=0.0-1.0
            'mosaic': [0.8, 1.0],             # float, default=1.0, range=0.0-1.0
            'mixup': [0.0, 0.15],             # float, default=0.0, range=0.0-1.0
            'copy_paste': [0.0, 0.1],         # float, default=0.0, range=0.0-1.0
            
            # ✅ LOSS FUNCTION PARAMETERS (Officially Documented)
            'box': [5.0, 7.5, 10.0],          # float, default=7.5 (box loss gain)
            'cls': [0.3, 0.5, 1.0],           # float, default=0.5 (class loss gain)
            'dfl': [1.0, 1.5, 2.0],           # float, default=1.5 (DFL loss gain)
            
            # ✅ TRAINING CONTROL PARAMETERS (Officially Documented)
            'warmup_epochs': [3.0, 5.0],      # float, default=3.0
            'warmup_momentum': [0.8, 0.9],    # float, default=0.8
            'warmup_bias_lr': [0.1, 0.2],     # float, default=0.1
            'cos_lr': [False, True],           # bool, default=False (cosine LR scheduler)
            
            # ✅ VALIDATION/INFERENCE PARAMETERS (Officially Documented) 
            'conf': [0.001, 0.01, 0.25],      # float, default=0.25 (confidence threshold)
            'iou': [0.6, 0.7, 0.8],           # float, default=0.7 (IoU threshold for NMS)
            
            # ✅ ADVANCED TRAINING PARAMETERS (Officially Documented)
            'patience': [50, 100, 150],       # int, default=100 (early stopping)
            'save_period': [-1, 10, 50],      # int, default=-1 (save checkpoint every N epochs)
            'amp': [True],                    # bool, default=True (Automatic Mixed Precision)
            'fraction': [0.8, 1.0],           # float, default=1.0 (dataset fraction)
            'close_mosaic': [0, 10, 15],      # int, default=10 (disable mosaic last N epochs)
            'multi_scale': [False, True],     # bool, default=False (multi-scale training)
            'dropout': [0.0, 0.1],            # float, default=0.0 (dropout rate)
            'nbs': [32, 64, 128],             # int, default=64 (nominal batch size)
        }
    
    def create_config_combinations(self, max_combinations=50, mode='priority'):
        """
        Create hyperparameter combinations with different strategies
        
        Args:
            max_combinations: Maximum number of configurations to generate
            mode: 'priority' (hand-picked configs) or 'grid' (systematic combinations) or 'random'
        """
        params = self.get_optimized_hyperparameters()
        
        if mode == 'priority':
            # Hand-picked priority combinations for medical imaging
            priority_configs = [
                # High accuracy configuration - VALIDATED PARAMETERS ONLY
                {
                    'epochs': 100, 'batch': 16, 'imgsz': 416,
                    'lr0': 0.01, 'lrf': 0.1, 'optimizer': 'AdamW',
                    'conf': 0.001, 'box': 10.0, 'cls': 0.5,
                    'hsv_h': 0.015, 'degrees': 5.0, 'mosaic': 1.0,
                    'amp': True, 'patience': 100
                },
                # Balanced configuration - VALIDATED PARAMETERS ONLY
                {
                    'epochs': 100, 'batch': 16, 'imgsz': 312,
                    'lr0': 0.01, 'lrf': 0.1, 'optimizer': 'SGD',
                    'conf': 0.01, 'box': 7.5, 'cls': 0.5,
                    'hsv_h': 0.020, 'degrees': 0.0, 'mosaic': 0.8,
                    'amp': True, 'patience': 100
                },
                # Fast training configuration - VALIDATED PARAMETERS ONLY
                {
                    'epochs': 100, 'batch': 32, 'imgsz': 208,
                    'lr0': 0.02, 'lrf': 0.2, 'optimizer': 'Adam',
                    'conf': 0.25, 'box': 7.5, 'cls': 0.5,
                    'hsv_h': 0.025, 'degrees': 10.0, 'mosaic': 1.0,
                    'amp': True, 'patience': 50
                }
            ]
            return priority_configs[:max_combinations]
        
        elif mode == 'grid':
            # Systematic grid search of key parameters
            key_params = {
                'epochs': params['epochs'],
                'lr0': params['lr0'], 
                'imgsz': params['imgsz'],
                'optimizer': params['optimizer'],
                'conf_thres': params['conf_thres']
            }
            
            import itertools
            combinations = []
            param_names = list(key_params.keys())
            param_values = [key_params[name] for name in param_names]
            
            for combo in itertools.product(*param_values):
                config = dict(zip(param_names, combo))
                # Add some fixed values for other parameters - VALIDATED ONLY
                config.update({
                    'batch': 16,
                    'box': 7.5,
                    'cls': 0.5,
                    'hsv_h': 0.015,
                    'degrees': 0.0,
                    'amp': True
                })
                combinations.append(config)
                
                if len(combinations) >= max_combinations:
                    break
            
            return combinations
        
        elif mode == 'full_grid':
            # EXHAUSTIVE grid search - WARNING: Can generate thousands of combinations!
            import itertools
            
            # Select key parameters for full grid - VALIDATED ONLY
            grid_params = {
                'epochs': params['epochs'][:2],      # [100, 150]
                'batch': params['batch'][:2],        # [8, 16] 
                'imgsz': params['imgsz'][:2],        # [640, 832]
                'lr0': params['lr0'][:2],            # [0.001, 0.01]
                'optimizer': params['optimizer'][:2], # ['SGD', 'Adam']
                'conf': params['conf'][:2],          # [0.001, 0.01] 
                'box': params['box'][:2],            # [5.0, 7.5]
                'cls': params['cls'][:2]             # [0.3, 0.5]
            }
            
            combinations = []
            param_names = list(grid_params.keys())
            param_values = [grid_params[name] for name in param_names]
            
            for combo in itertools.product(*param_values):
                config = dict(zip(param_names, combo))
                combinations.append(config)
                
                if len(combinations) >= max_combinations:
                    break
            
            print(f"🔢 Generated {len(combinations)} grid combinations")
            return combinations
        
        else:  # mode == 'random'
            # Random combinations for exploration
            import random
            random_configs = []
            for _ in range(max_combinations):
                config = {}
                for param, values in params.items():
                    if param != 'models':
                        if isinstance(values, list) and len(values) > 0:
                            config[param] = random.choice(values)
                random_configs.append(config)
            
            return random_configs
    
    def train_model_with_config(self, model_name, config, run_name):
        """
        Train a YOLO model with specific hyperparameter configuration
        """
        try:
            print(f"\n🚀 Training {model_name} with config: {run_name}")
            print(f"Key parameters: epochs={config.get('epochs')}, "
                  f"lr0={config.get('lr0')}, imgsz={config.get('imgsz')}")
            
            # Load model
            model = YOLO(model_name)
            
            # Prepare training arguments
            train_args = {
                'data': self.data_path,
                'device': self.device,
                'workers': self.workers,
                'project': 'lung_nodule_detection',
                'name': run_name,
                'exist_ok': True,
                'save': True,
                'save_period': 50,
                'cache': True,
                'verbose': False,
                **config  # Unpack all hyperparameters
            }
            
            # Train model
            results = model.train(**train_args)
            
            # Extract key metrics
            metrics = {
                'model': model_name,
                'run_name': run_name,
                'config': config,
                'map50': results.results_dict.get('metrics/mAP50(B)', 0),
                'map50_95': results.results_dict.get('metrics/mAP50-95(B)', 0),
                'precision': results.results_dict.get('metrics/precision(B)', 0),
                'recall': results.results_dict.get('metrics/recall(B)', 0),
                'box_loss': results.results_dict.get('train/box_loss', float('inf')),
                'cls_loss': results.results_dict.get('train/cls_loss', float('inf')),
                'dfl_loss': results.results_dict.get('train/dfl_loss', float('inf')),
                'fitness': results.fitness,
                'timestamp': datetime.now().isoformat()
            }
            
            self.results_log.append(metrics)
            return metrics
            
        except Exception as e:
            print(f"❌ Error training {model_name} with {run_name}: {str(e)}")
            return None
    
    def run_hyperparameter_tuning(self, model_versions=['yolov8'], mode='priority', max_combinations=15):
        """
        Run comprehensive hyperparameter tuning
        
        Args:
            model_versions: List of YOLO versions to test
            mode: 'priority' (hand-picked), 'grid' (systematic), 'full_grid' (exhaustive), 'random'
            max_combinations: Maximum number of configurations per model
        """
        print("🔬 Starting YOLO Hyperparameter Tuning for Lung Nodule Detection")
        print(f"📊 Mode: {mode.upper()}")
        print("=" * 70)
        
        params = self.get_optimized_hyperparameters()
        configs = self.create_config_combinations(max_combinations=max_combinations, mode=mode)
        
        print(f"🎯 Generated {len(configs)} configurations")
        
        # Show example configurations
        print(f"\n📋 Example configurations:")
        for i, config in enumerate(configs[:3]):
            print(f"  Config {i+1}: epochs={config.get('epochs')}, lr0={config.get('lr0')}, "
                  f"imgsz={config.get('imgsz')}, optimizer={config.get('optimizer')}")
        
        if mode == 'full_grid':
            total_possible = 1
            for param, values in params.items():
                if param != 'models' and isinstance(values, list):
                    total_possible *= len(values)
            print(f"⚠️  Full grid would generate {total_possible:,} combinations!")
            print(f"   Limited to {len(configs)} for practical training time")
        
        model_count = sum(len(params['models'][v]) for v in model_versions if v in params['models'])
        total_experiments = model_count * len(configs)
        experiment_count = 0
        
        print(f"\n🚀 Total experiments: {total_experiments}")
        estimated_hours = total_experiments * 0.75  # Rough estimate
        print(f"⏱️  Estimated time: {estimated_hours:.1f} hours")
        
        for version in model_versions:
            if version not in params['models']:
                print(f"⚠️  Skipping {version} - not available")
                continue
                
            for model_name in params['models'][version]:
                for i, config in enumerate(configs):
                    experiment_count += 1
                    run_name = f"{version}_{model_name.replace('.pt', '')}_{mode}_{i+1}"
                    
                    print(f"\n📊 Experiment {experiment_count}/{total_experiments}")
                    
                    result = self.train_model_with_config(model_name, config, run_name)
                    
                    if result:
                        print(f"✅ Completed: mAP50={result['map50']:.4f}, "
                              f"mAP50-95={result['map50_95']:.4f}")
                    
                    # Save intermediate results
                    self.save_results()
        
        print(f"\n🎉 Hyperparameter tuning completed using {mode} mode!")
        self.analyze_results()
        
    def save_results(self):
        """Save results to CSV and YAML files"""
        if not self.results_log:
            return
            
        # Save to CSV
        df = pd.DataFrame(self.results_log)
        df.to_csv('yolo_hyperparameter_results.csv', index=False)
        
        # Save detailed results to YAML
        with open('yolo_hyperparameter_results.yaml', 'w') as f:
            yaml.dump(self.results_log, f, default_flow_style=False)
    
    def analyze_results(self):
        """Analyze results and provide recommendations"""
        if not self.results_log:
            print("No results to analyze")
            return
        
        df = pd.DataFrame(self.results_log)
        
        print("\n📈 RESULTS ANALYSIS")
        print("=" * 50)
        
        # Best models by different metrics
        best_map50 = df.loc[df['map50'].idxmax()]
        best_map50_95 = df.loc[df['map50_95'].idxmax()]
        best_precision = df.loc[df['precision'].idxmax()]
        best_recall = df.loc[df['recall'].idxmax()]
        
        print(f"\n🏆 Best mAP50: {best_map50['model']} - {best_map50['run_name']}")
        print(f"   Score: {best_map50['map50']:.4f}")
        print(f"   Key params: epochs={best_map50['config'].get('epochs')}, "
              f"lr0={best_map50['config'].get('lr0')}")
        
        print(f"\n🏆 Best mAP50-95: {best_map50_95['model']} - {best_map50_95['run_name']}")
        print(f"   Score: {best_map50_95['map50_95']:.4f}")
        
        print(f"\n🏆 Best Precision: {best_precision['model']} - {best_precision['run_name']}")
        print(f"   Score: {best_precision['precision']:.4f}")
        
        print(f"\n🏆 Best Recall: {best_recall['model']} - {best_recall['run_name']}")
        print(f"   Score: {best_recall['recall']:.4f}")
        
        # Parameter analysis
        print(f"\n📊 Parameter Impact Analysis:")
        
        # Group by model version
        for model_type in ['yolov8', 'yolov10', 'yolov11']:
            model_results = df[df['model'].str.contains(model_type)]
            if not model_results.empty:
                avg_map50 = model_results['map50'].mean()
                print(f"   {model_type.upper()}: Average mAP50 = {avg_map50:.4f}")
        
        # Recommendations
        print(f"\n💡 RECOMMENDATIONS FOR LUNG NODULE DETECTION:")
        print("=" * 50)
        print("1. Use higher resolution (imgsz=1024) for better small nodule detection")
        print("2. Lower confidence threshold (0.001-0.01) to catch subtle nodules")
        print("3. Increase box_gain (0.05-0.1) for better small object localization")
        print("4. Use AdamW optimizer with learning rate 0.01 for medical imaging")
        print("5. Train for more epochs (150-200) as medical data is complex")
        print("6. Conservative data augmentation to preserve medical image characteristics")

# Example usage with different modes
if __name__ == "__main__":
    # Initialize tuner
    tuner = YOLOHyperparameterTuner(
        data_path='/kaggle/working/data.yml',
        device=0,
        workers=10
    )
    
    # # 🎯 OPTION 1: Priority Mode (RECOMMENDED for quick results)
    # # Uses hand-picked configurations optimized for medical imaging
    # tuner.run_hyperparameter_tuning(
    #     model_versions=['yolov8'], 
    #     mode='priority', 
    #     max_combinations=3  # Quick: 3 configs × 3 models = 9 experiments (~3 hours)
    # )
    
    # # 🔍 OPTION 2: Systematic Grid Search (for thorough exploration) 
    # # Tests key parameter combinations systematically
    # # tuner.run_hyperparameter_tuning(
    # #     model_versions=['yolov8'], 
    # #     mode='grid', 
    # #     max_combinations=27  # 3×3×3 = 27 combinations (~20 hours)
    # # )
    
    # # 🌐 OPTION 3: Full Grid Search (WARNING: Very time-consuming!)
    # # Tests ALL combinations from your parameter lists
    # # tuner.run_hyperparameter_tuning(
    # #     model_versions=['yolov8'], 
    # #     mode='full_grid', 
    # #     max_combinations=100  # Limited to prevent excessive runtime
    # # )
    
    # # 🎲 OPTION 4: Random Search (for exploration)
    # # Randomly samples from all parameter combinations
    # # tuner.run_hyperparameter_tuning(
    # #     model_versions=['yolov8'], 
    # #     mode='random', 
    # #     max_combinations=20
    # # )
    
    # # 🚀 QUICK SINGLE TEST with VALIDATED parameters:
    # """
    # model = YOLO('yolov8s.pt')
    # results = model.train(
    #     data='/kaggle/working/data.yml',
    #     epochs=150,
    #     imgsz=832,
    #     batch=16,           # CORRECTED: was batch_size
    #     lr0=0.01,
    #     optimizer='AdamW',
    #     conf=0.001,         # CORRECTED: was conf_thres  
    #     box=10.0,           # CORRECTED: was box_gain
    #     cls=0.5,            # CORRECTED: was cls_gain
    #     device=0,
    #     workers=8,
    #     project='lung_nodule_detection',
    #     name='single_best_config'
    # )
    # """

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
tuner.run_hyperparameter_tuning(
    model_versions=['yolov11'], 
    mode='priority', 
    max_combinations=3
)


🔬 Starting YOLO Hyperparameter Tuning for Lung Nodule Detection
📊 Mode: PRIORITY
🎯 Generated 3 configurations

📋 Example configurations:
  Config 1: epochs=100, lr0=0.01, imgsz=416, optimizer=AdamW
  Config 2: epochs=100, lr0=0.01, imgsz=312, optimizer=SGD
  Config 3: epochs=100, lr0=0.02, imgsz=208, optimizer=Adam

🚀 Total experiments: 9
⏱️  Estimated time: 6.8 hours

📊 Experiment 1/9

🚀 Training yolo11n.pt with config: yolov11_yolo11n_priority_1
Key parameters: epochs=100, lr0=0.01, imgsz=416
Ultralytics 8.3.202 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=10.0, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, ep

/usr/local/lib/python3.11/dist-packages/ultralytics/engine/validator.py:288: RuntimeWarning: invalid value encountered in greater_equal
  matches = np.nonzero(iou >= threshold)  # IoU > threshold and classes match



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100       1.2G      3.223      2.444      1.206         11        416: 100% ━━━━━━━━━━━━ 10/10 7.7it/s 1.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 11.9it/s 0.2s
                   all         33         36          0          0          0          0

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      5/100      1.22G      3.011      2.184      1.203         13        416: 100% ━━━━━━━━━━━━ 10/10 7.9it/s 1.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 13.4it/s 0.1s
                   all         33         36          0          0          0          0


/usr/local/lib/python3.11/dist-packages/ultralytics/engine/validator.py:288: RuntimeWarning: invalid value encountered in greater_equal
  matches = np.nonzero(iou >= threshold)  # IoU > threshold and classes match



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      6/100      1.24G      3.168      2.246      1.206         10        416: 100% ━━━━━━━━━━━━ 10/10 8.1it/s 1.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 11.4it/s 0.2s
                   all         33         36          0          0          0          0

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      7/100      1.25G      3.058      2.097       1.16         10        416: 100% ━━━━━━━━━━━━ 10/10 8.5it/s 1.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 11.8it/s 0.2s
                   all         33         36          0          0          0          0

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      8/100      1.27G      3.203      2.209      1.215         16        416: 100% ━━━━━━━━━━━━ 10/1

...

                   all         33         36      0.709      0.813      0.809      0.456
Speed: 0.1ms preprocess, 6.2ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to /kaggle/working/lung_nodule_detection/yolov11_yolo11m_priority_1
✅ Completed: mAP50=0.8091, mAP50-95=0.4560

📊 Experiment 8/9

🚀 Training yolo11m.pt with config: yolov11_yolo11m_priority_2
Key parameters: epochs=100, lr0=0.01, imgsz=312
Ultralytics 8.3.202 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.01, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscr

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all         33         36      0.941      0.972      0.981      0.667
Speed: 0.1ms preprocess, 4.5ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /kaggle/working/lung_nodule_detection/yolov11_yolo11m_priority_2
✅ Completed: mAP50=0.9812, mAP50-95=0.6669

📊 Experiment 9/9

🚀 Training yolo11m.pt with config: yolov11_yolo11m_priority_3
Key parameters: epochs=100, lr0=0.02, imgsz=208
Ultralytics 8.3.202 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.25, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchsc

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all         33         36      0.848      0.306      0.578       0.35
Speed: 0.0ms preprocess, 2.2ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /kaggle/working/lung_nodule_detection/yolov11_yolo11m_priority_3
✅ Completed: mAP50=0.5784, mAP50-95=0.3504

🎉 Hyperparameter tuning completed using priority mode!

📈 RESULTS ANALYSIS

🏆 Best mAP50: yolo11s.pt - yolov11_yolo11s_priority_2
   Score: 0.9848
   Key params: epochs=100, lr0=0.01

🏆 Best mAP50-95: yolo11m.pt - yolov11_yolo11m_priority_2
   Score: 0.6669

🏆 Best Precision: yolo11n.pt - yolov11_yolo11n_priority_3
   Score: 1.0000

🏆 Best Recall: yolo11m.pt - yolov11_yolo11m_priority_2
   Score: 0.9722

📊 Parameter Impact Analysis:

💡 RECOMMENDATIONS FOR LUNG NODULE DETECTION:
1. Use higher resolution (imgsz=1024) for better small nodule detection
2. Lower confidence threshold (0.001-0.01) to catch subtle nodules
3. Increase box_gain (0.05-0.1) for better small object localization
4. Use AdamW